# VGG16 and ResNet50 validation

This notebook compares the old and new ResNet global average pooling paths and benchmarks full ResNet50 inference on CPU and, when available, on CUDA. It also compares all 1,000 logits against local DeZero for the same input and official weights.

The pooling benchmark runs offline. For the pretrained check, install Pillow, point `REFERENCE_DIR` to the DeZero checkout, and set `RUN_OFFICIAL_WEIGHTS = True`. The official archives are large and are loaded one model at a time.

## ResNet global average pooling

The baseline uses full-window average pooling and reshape. The optimized path calls MyDeZero's current helper, which reduces over the spatial axes. The cases vary batch size, channel count, and spatial extent. Each case checks the output and input gradient before timing.

Measurements exclude input creation, gradient recording, and warm-up. Each row reports the median of seven per-batch averages, alternating which implementation runs first. GPU timings synchronize the CUDA stream after each call and include host dispatch and synchronization. The GPU section runs when CuPy can access a CUDA device.

In [1]:
import platform
from statistics import median
from time import perf_counter_ns

import numpy as np

from mydezero import Variable, cuda, no_grad
from mydezero.functions import average_pooling
from mydezero.models import _global_average_pooling_2d


def previous_pooling(x):
    n, c, h, w = x.shape
    return average_pooling(x, (h, w), stride=1).reshape(n, c)


def time_calls(operation, x, calls, synchronize):
    start = perf_counter_ns()
    for _ in range(calls):
        operation(x)
        if synchronize is not None:
            synchronize()
    return (perf_counter_ns() - start) / calls / 1e6


def benchmark_pooling(xp, synchronize=None):
    shapes = [
        (1, 2048, 7, 7),
        (8, 2048, 7, 7),
        (16, 2048, 7, 7),
        (1, 2048, 4, 4),
        (1, 2048, 14, 14),
        (1, 2048, 7, 11),
        (1, 512, 7, 7),
        (4, 512, 14, 14),
    ]
    calls = 50
    repeats = 7
    operations = (previous_pooling, _global_average_pooling_2d)
    rng = np.random.default_rng(71)
    print(f"FP32 forward: median of {repeats} batches of {calls} calls after warm-up")
    print(f"{'Input shape':<20} {'Previous (ms)':>14} {'Direct (ms)':>14} {'Speedup':>10}")
    for shape in shapes:
        data = xp.asarray(rng.standard_normal(shape, dtype=np.float32))
        upstream = xp.asarray(rng.standard_normal(shape[:2], dtype=np.float32))
        before, after = Variable(data), Variable(data)
        expected = previous_pooling(before)
        actual = _global_average_pooling_2d(after)
        (expected * upstream).sum().backward()
        (actual * upstream).sum().backward()
        xp.testing.assert_allclose(actual.data, expected.data, rtol=1e-6, atol=1e-7)
        xp.testing.assert_allclose(after.grad.data, before.grad.data, rtol=1e-6, atol=1e-7)

        samples = {operation: [] for operation in operations}
        with no_grad():
            for operation in operations:
                time_calls(operation, before, 10, synchronize)
            for trial in range(repeats):
                order = operations if trial % 2 == 0 else operations[::-1]
                for operation in order:
                    samples[operation].append(time_calls(operation, before, calls, synchronize))
        baseline, direct = (median(samples[operation]) for operation in operations)
        print(f"{str(shape):<20} {baseline:14.3f} {direct:14.3f} {baseline / direct:9.2f}x")


print(f"CPU / {platform.system()} / Python {platform.python_version()} / NumPy {np.__version__}")
benchmark_pooling(np)

CPU / Windows / Python 3.12.16 / NumPy 1.26.4
FP32 forward: median of 7 batches of 50 calls after warm-up
Input shape           Previous (ms)    Direct (ms)    Speedup
(1, 2048, 7, 7)               0.508          0.052      9.68x
(8, 2048, 7, 7)               5.875          0.364     16.14x
(16, 2048, 7, 7)             16.656          0.697     23.88x
(1, 2048, 4, 4)               0.206          0.036      5.68x
(1, 2048, 14, 14)             3.384          0.151     22.46x
(1, 2048, 7, 11)              0.782          0.070     11.15x
(1, 512, 7, 7)                0.188          0.021      8.75x
(4, 512, 14, 14)              3.391          0.144     23.47x


In [2]:
def benchmark_cuda(benchmark):
    if not cuda.gpu_enable:
        print("GPU benchmark skipped: CuPy is not installed.")
        return

    cp = cuda.cupy
    try:
        device_count = cp.cuda.runtime.getDeviceCount()
    except cp.cuda.runtime.CUDARuntimeError as error:
        print(f"GPU benchmark skipped: {error}")
        return
    if device_count == 0:
        print("GPU benchmark skipped: no CUDA device.")
        return

    device = cp.cuda.Device()
    name = cp.cuda.runtime.getDeviceProperties(device.id)["name"].decode()
    print(f"GPU / {name} / CuPy {cp.__version__} / synchronized wall time")
    benchmark(cp, cp.cuda.get_current_stream().synchronize)


benchmark_cuda(benchmark_pooling)

GPU / NVIDIA GeForce RTX 5060 Laptop GPU / CuPy 13.6.0 / synchronized wall time
FP32 forward: median of 7 batches of 50 calls after warm-up
Input shape           Previous (ms)    Direct (ms)    Speedup
(1, 2048, 7, 7)               0.134          0.082      1.63x
(8, 2048, 7, 7)               0.136          0.099      1.37x
(16, 2048, 7, 7)              0.194          0.172      1.13x
(1, 2048, 4, 4)               0.112          0.086      1.30x
(1, 2048, 14, 14)             0.129          0.098      1.31x
(1, 2048, 7, 11)              0.109          0.079      1.38x
(1, 512, 7, 7)                0.116          0.085      1.36x
(4, 512, 14, 14)              0.121          0.080      1.51x


## Official weight acceptance

Set `REFERENCE_DIR` to the checkout root that contains the `dezero/` source folder. Use `WEIGHTS_DIR` if the original `vgg16.npz` and `resnet50.npz` files are already available locally. Otherwise MyDeZero retrieves them from the official release and reuses its download cache. To check only one archive, set `MODELS_TO_CHECK` to a single name, such as `("ResNet50",)`.

Both frameworks run in inference mode on CPU, load the same archive and receive the same batch. With no `IMAGE_PATH`, the check uses a seeded RGB image. This measures numerical agreement between the implementations. The latency above measures the pooling operator only. Saved acceptance results show which archives were tested with the check enabled.

In [3]:
from pathlib import Path

from mydezero import test_mode
from mydezero.models import ResNet50, VGG16
from mydezero.utils import get_file

RUN_OFFICIAL_WEIGHTS = False
RUN_RESNET50_BENCHMARK = False
MODELS_TO_CHECK = ("VGG16", "ResNet50")
REFERENCE_DIR = Path("reference/deep-learning-from-scratch-3-master-cn")
WEIGHTS_DIR = None
IMAGE_PATH = None


def validation_image():
    from PIL import Image

    if IMAGE_PATH is None:
        pixels = np.random.default_rng(2026).integers(0, 256, (256, 256, 3), dtype=np.uint8)
        return Image.fromarray(pixels)
    with Image.open(Path(IMAGE_PATH).expanduser()) as image:
        return image.convert("RGB")


def pretrained_batch(image):
    return np.ascontiguousarray(VGG16.preprocess(image)[None, ...])


def weight_archive(url):
    if WEIGHTS_DIR is None:
        return Path(get_file(url))
    path = Path(WEIGHTS_DIR).expanduser() / url.rsplit("/", 1)[-1]
    if not path.is_file():
        raise FileNotFoundError(path)
    return path

In [4]:
import sys



def official_weight_check():
    reference_root = Path(REFERENCE_DIR).expanduser()
    if not reference_root.is_absolute():
        reference_root = next(
            (root / reference_root for root in (Path.cwd(), *Path.cwd().parents)
             if (root / reference_root / "dezero" / "models.py").is_file()),
            reference_root,
        )
    reference_root = reference_root.resolve()
    if not (reference_root / "dezero" / "models.py").is_file():
        raise FileNotFoundError(f"Set REFERENCE_DIR to the DeZero checkout: {reference_root}")

    # The local DeZero edition uses np.int in a default argument at import time.
    sys.path.insert(0, str(reference_root))
    needs_int_alias = "int" not in vars(np)
    if needs_int_alias:
        np.int = int
    try:
        import dezero
        from dezero import models as reference_models
    finally:
        if needs_int_alias:
            del np.int
        sys.path.pop(0)
    if not Path(reference_models.__file__).resolve().is_relative_to(reference_root):
        raise RuntimeError("Restart the kernel to import the selected DeZero checkout.")

    image = validation_image()
    batch = pretrained_batch(image)
    np.testing.assert_allclose(batch[0], reference_models.VGG16.preprocess(image))

    def infer(model_class, weights_path, eval_context, no_grad_context):
        model = model_class()
        model.load_weights(str(weights_path))
        with eval_context(), no_grad_context():
            return model(batch).data

    models = {
        "VGG16": (VGG16, reference_models.VGG16, VGG16.WEIGHTS_PATH),
        "ResNet50": (ResNet50, reference_models.ResNet50, ResNet50.WEIGHTS_PATH.format(50)),
    }
    print("Model | archive | logits shape | max absolute error")
    for name in MODELS_TO_CHECK:
        model_class, ref_class, url = models[name]
        weights_path = weight_archive(url)
        actual = infer(model_class, weights_path, test_mode, no_grad)
        expected = infer(ref_class, weights_path, dezero.test_mode, dezero.no_grad)
        assert actual.shape == expected.shape == (1, 1000)
        assert np.isfinite(actual).all() and np.isfinite(expected).all()
        error = np.max(np.abs(actual - expected))
        print(f"{model_class.__name__} | {weights_path.name} | {actual.shape} | {error:.6g}")
        np.testing.assert_allclose(actual, expected, rtol=1e-4, atol=1e-3)
    print("PASS: MyDeZero logits agree with DeZero.")


if RUN_OFFICIAL_WEIGHTS:
    official_weight_check()
else:
    print("Official weight check skipped. Set RUN_OFFICIAL_WEIGHTS = True to enable it.")

Model | archive | logits shape | max absolute error
VGG16 | vgg16.npz | (1, 1000) | 0
ResNet50 | resnet50.npz | (1, 1000) | 2.86102e-06
PASS: MyDeZero logits agree with DeZero.


## ResNet50 full-model inference

Set `RUN_RESNET50_BENCHMARK = True` to benchmark both pooling paths with the same official ResNet50 weights and FP32 input of shape `(1, 3, 224, 224)`. Set `WEIGHTS_DIR` to use a local archive. MyDeZero loads the archive directly, so this benchmark can run with the DeZero reference check disabled.

The benchmark checks all 1,000 logits, warms up both paths, then alternates the timing order across 11 forward passes per path on each device. The median and range are wall-clock times. Weight loading, preprocessing, and device transfers are outside the timed region. CUDA timings include stream synchronization.

In [5]:
import os
from unittest.mock import patch

import mydezero.models as model_module


def benchmark_resnet50(xp, synchronize=None):
    model = ResNet50()
    model.load_weights(weight_archive(ResNet50.WEIGHTS_PATH.format(50)))
    if xp is not np:
        model.to_gpu()
    batch = xp.asarray(pretrained_batch(validation_image()))
    operations = (previous_pooling, model_module._global_average_pooling_2d)
    samples = {operation: [] for operation in operations}
    outputs = []
    trials = 11

    with test_mode(), no_grad():
        for operation in operations:
            with patch.object(model_module, "_global_average_pooling_2d", operation):
                outputs.append(model(batch).data)
                time_calls(model, batch, 3, synchronize)
        for trial in range(trials):
            order = operations if trial % 2 == 0 else operations[::-1]
            for operation in order:
                with patch.object(model_module, "_global_average_pooling_2d", operation):
                    samples[operation].append(time_calls(model, batch, 1, synchronize))

    before, after = outputs
    assert before.shape == after.shape == (1, 1000)
    assert bool(xp.isfinite(before).all() & xp.isfinite(after).all())
    xp.testing.assert_allclose(after, before, rtol=1e-4, atol=1e-3)
    maximum_error = float(xp.max(xp.abs(after - before)))
    print(f"Input {batch.shape} / float32 / {trials} timed forwards per pooling path")
    for label, operation in zip(("Previous pooling", "Direct pooling"), operations):
        times = samples[operation]
        print(f"{label:<18} {median(times):8.3f} ms median "
              f"({min(times):.3f} to {max(times):.3f} ms)")
    print(f"Max absolute logit difference: {maximum_error:.6g}")


if RUN_RESNET50_BENCHMARK:
    threads = os.environ.get("OPENBLAS_NUM_THREADS", "runtime default")
    print(f"CPU / {platform.system()} / NumPy {np.__version__} / OPENBLAS_NUM_THREADS={threads}")
    benchmark_resnet50(np)
    benchmark_cuda(benchmark_resnet50)
else:
    print("ResNet50 full-model benchmark skipped. Set RUN_RESNET50_BENCHMARK = True to enable it.")

CPU / Windows / NumPy 1.26.4 / OPENBLAS_NUM_THREADS=4
Input (1, 3, 224, 224) / float32 / 11 timed forwards per pooling path
Previous pooling    216.107 ms median (172.581 to 244.200 ms)
Direct pooling      213.862 ms median (175.650 to 277.642 ms)
Max absolute logit difference: 2.86102e-06
GPU / NVIDIA GeForce RTX 5060 Laptop GPU / CuPy 13.6.0 / synchronized wall time
Input (1, 3, 224, 224) / float32 / 11 timed forwards per pooling path
Previous pooling     28.867 ms median (20.386 to 32.829 ms)
Direct pooling       26.406 ms median (21.015 to 30.200 ms)
Max absolute logit difference: 1.4614e-06
